# FABRIC Visualization Suite — Demo

This notebook demonstrates `FabVis`, the unified FABRIC visualization tool.
It uses **mock data** so you can test the UI without a live FABRIC connection.

**Sections:**
1. **Interactive Tool** — Launch FabVis (Editor + Geographic views, image export)
2. **Static Graph Image** — Generate a topology diagram from a slice object
3. **Static Map Image** — Generate a geographic map from a slice object

## Setup — Mock Data

Mock classes that mimic the FABlib API so the visualizer works without a real FABRIC connection.

In [ ]:
class MockInterface:
    def __init__(self, name, ip=None, vlan=None, mac=None, bandwidth=None, node=None, network=None, site=None):
        self._name = name; self._ip = ip; self._vlan = vlan; self._mac = mac
        self._bw = bandwidth; self._node = node; self._network = network; self._site = site
    def get_name(self): return self._name
    def get_ip_addr(self): return self._ip
    def get_vlan(self): return self._vlan
    def get_mac(self): return self._mac
    def get_bandwidth(self): return self._bw
    def get_site(self): return self._site
    def get_physical_os_interface_name(self): return "eth1"
    def get_reservation_state(self): return "Active"
    def get_subnet(self): return "10.0.0.0/24" if self._ip else None
    def get_node(self): return self._node
    def get_network(self): return self._network
    def set_ip_addr(self, addr=None, mode=None): self._ip = str(addr) if addr else None
    def set_vlan(self, vlan=None): self._vlan = vlan
    def set_network(self, net): self._network = net
    def delete(self): self._network = None

class MockComponent:
    def __init__(self, name, model, interfaces=None, node=None):
        self._name = name; self._model = model
        self._interfaces = interfaces or []; self._node = node
    def get_name(self): return self._name
    def get_model(self): return self._model
    def get_type(self): return self._model.split("_")[0]
    def get_details(self): return f"{self._model} component"
    def get_pci_addr(self): return "0000:41:00.0"
    def get_numa_node(self): return "0"
    def get_device_name(self): return None
    def get_interfaces(self): return self._interfaces
    def get_node(self): return self._node
    def delete(self):
        if self._node:
            self._node._components = [c for c in self._node._components if c is not self]

class MockNode:
    _iface_counter = 0
    def __init__(self, name, site="RENC", cores=4, ram=16, disk=100,
                 image="default_rocky_9", components=None, interfaces=None, state="Active"):
        self._name = name; self._site = site; self._cores = cores; self._ram = ram
        self._disk = disk; self._image = image; self._state = state
        self._components = components or []; self._interfaces = interfaces or []
    def get_name(self): return self._name
    def get_site(self): return self._site
    def get_host(self): return f"{self._site.lower()}-w1.fabric-testbed.net"
    def get_cores(self): return self._cores
    def get_ram(self): return self._ram
    def get_disk(self): return self._disk
    def get_image(self): return self._image
    def get_management_ip(self): return f"2001:db8::{hash(self._name) % 256}"
    def get_reservation_state(self): return self._state
    def get_username(self): return "rocky"
    def get_components(self): return self._components
    def get_component(self, name):
        for c in self._components:
            if c.get_name() == name: return c
        raise Exception(f"Component '{name}' not found")
    def get_interfaces(self): return self._interfaces
    def set_site(self, site): self._site = site
    def set_capacities(self, cores=2, ram=2, disk=10):
        self._cores = cores; self._ram = ram; self._disk = disk
    def set_image(self, image, username=None, image_type="qcow2"): self._image = image
    def add_component(self, model=None, name=None, user_data={}):
        MockNode._iface_counter += 1
        cname = name or f"{model.lower().split('_')[0]}_{len(self._components)+1}"
        iface = MockInterface(f"{self._name}-{cname}-p0", site=self._site)
        iface._node = self
        cmp = MockComponent(cname, model, [iface], node=self)
        self._components.append(cmp)
        self._interfaces.append(iface)
        return cmp
    def delete(self):
        if hasattr(self, '_slice') and self._slice:
            self._slice._nodes = [n for n in self._slice._nodes if n is not self]

class MockFacilityPort:
    def __init__(self, name, site, interfaces=None):
        self._name = name; self._site = site; self._ifaces = interfaces or []
    def get_name(self): return self._name
    def get_site(self): return self._site
    def get_model(self): return "Facility_Port"
    def get_host(self): return None
    def get_cores(self): return None
    def get_ram(self): return None
    def get_disk(self): return None
    def get_image(self): return None
    def get_management_ip(self): return None
    def get_reservation_state(self): return "Active"
    def get_username(self): return None
    def get_components(self): return []
    def get_interfaces(self): return self._ifaces

class MockNetworkService:
    def __init__(self, name, net_type="L2Bridge", interfaces=None, subnet=None, gateway=None, site=None):
        self._name = name; self._type = net_type
        self._interfaces = interfaces or []; self._subnet = subnet; self._gw = gateway; self._site = site
    def get_name(self): return self._name
    def get_type(self): return self._type
    def get_layer(self): return "L2" if "L2" in self._type else "L3"
    def get_site(self): return self._site or "RENC"
    def get_subnet(self): return self._subnet
    def get_gateway(self): return self._gw
    def get_interfaces(self): return self._interfaces
    def add_interface(self, iface): self._interfaces.append(iface); iface._network = self
    def set_subnet(self, subnet): self._subnet = str(subnet)
    def set_gateway(self, gw): self._gw = str(gw)
    def delete(self):
        for iface in self._interfaces: iface._network = None
        if hasattr(self, '_slice') and self._slice:
            self._slice._networks = [n for n in self._slice._networks if n is not self]

class MockSlice:
    def __init__(self, name, slice_id=None, state="StableOK",
                 nodes=None, networks=None, facilities=None):
        self._name = name; self._id = slice_id or f"id-{name}"
        self._state = state; self._nodes = nodes or []
        self._networks = networks or []; self._facilities = facilities or []
        for n in self._nodes: n._slice = self
        for net in self._networks: net._slice = self
    def get_name(self): return self._name
    def get_slice_id(self): return self._id
    def get_state(self): return self._state
    def get_lease_end(self): return "2026-03-15 00:00:00 +0000"
    def get_nodes(self): return self._nodes
    def get_node(self, name):
        for n in self._nodes:
            if n.get_name() == name: return n
        raise Exception(f"Node '{name}' not found")
    def get_network_services(self): return self._networks
    def get_facilities(self): return self._facilities
    def get_interfaces(self): return []
    def get_components(self): return []
    def add_node(self, name, site=None, cores=2, ram=8, disk=10,
                 image=None, instance_type=None, host=None, user_data={},
                 avoid=[], validate=False, raise_exception=False):
        node = MockNode(name, site=site or "RENC", cores=cores, ram=ram,
                        disk=disk, image=image or "default_rocky_9", state="Nascent")
        node._slice = self
        self._nodes.append(node)
        return node
    def add_l2network(self, name=None, interfaces=[], type=None, subnet=None,
                      gateway=None, user_data={}):
        net_type = type or "L2Bridge"
        net = MockNetworkService(name, net_type, list(interfaces), site="RENC")
        net._slice = self
        for iface in interfaces: iface._network = net
        self._networks.append(net)
        return net
    def add_l3network(self, name=None, interfaces=[], type="IPv4",
                      user_data={}, technology=None, subnet=None, site=None):
        type_map = {"IPv4": "FABNetv4", "IPv6": "FABNetv6",
                    "IPv4Ext": "FABNetv4Ext", "IPv6Ext": "FABNetv6Ext", "L3VPN": "L3VPN"}
        net_type = type_map.get(type, f"FABNet{type}")
        net = MockNetworkService(name, net_type, list(interfaces), site=site or "RENC")
        net._slice = self
        for iface in interfaces: iface._network = net
        self._networks.append(net)
        return net
    def submit(self, **kwargs): self._state = "Configuring"; return self._id
    def modify(self, **kwargs): return self._id
    def delete(self): self._nodes.clear(); self._networks.clear(); self._state = "Dead"

class MockFablib:
    def __init__(self, slices=None):
        self._slices = {s.get_name(): s for s in (slices or [])}
    def get_slices(self): return list(self._slices.values())
    def get_slice(self, name):
        if name not in self._slices: raise Exception(f"Slice '{name}' not found")
        return self._slices[name]
    def new_slice(self, name):
        s = MockSlice(name, state="Nascent")
        self._slices[name] = s
        return s

print("Mock classes defined.")

In [ ]:
# ── Slice 1: web_application ──
fe_nic_iface   = MockInterface("frontend-nic1-p0", ip="10.0.0.1", vlan="100", mac="aa:bb:cc:00:00:01", site="RENC")
be_nic1_iface  = MockInterface("backend-nic1-p0",  ip="10.0.0.2", vlan="100", mac="aa:bb:cc:00:00:02", bandwidth=100, site="RENC")
be_nic2_iface  = MockInterface("backend-nic2-p0",  ip="10.1.0.1", mac="aa:bb:cc:00:00:03", site="RENC")
db_nic_iface   = MockInterface("database-nic1-p0", ip="10.1.0.2", mac="aa:bb:cc:00:00:04", site="STAR")

fe_nic  = MockComponent("nic1", "NIC_Basic", [fe_nic_iface])
be_nic1 = MockComponent("nic1", "NIC_ConnectX_6", [be_nic1_iface])
be_nic2 = MockComponent("nic2", "NIC_Basic", [be_nic2_iface])
be_gpu  = MockComponent("gpu1", "GPU_A40", [])
db_nic  = MockComponent("nic1", "NIC_Basic", [db_nic_iface])
db_nvme = MockComponent("nvme1", "NVME_P4510", [])

frontend = MockNode("frontend", "RENC", cores=4, ram=8, disk=50,
                    components=[fe_nic], interfaces=[fe_nic_iface])
backend  = MockNode("backend", "RENC", cores=8, ram=32, disk=100,
                    components=[be_nic1, be_nic2, be_gpu], interfaces=[be_nic1_iface, be_nic2_iface])
database = MockNode("database", "STAR", cores=8, ram=64, disk=500,
                    components=[db_nic, db_nvme], interfaces=[db_nic_iface])

for iface in [fe_nic_iface]: iface._node = frontend
for iface in [be_nic1_iface, be_nic2_iface]: iface._node = backend
for iface in [db_nic_iface]: iface._node = database

app_bridge = MockNetworkService("app_bridge", "L2Bridge",
                                [fe_nic_iface, be_nic1_iface], subnet="10.0.0.0/24", site="RENC")
mgmt_net   = MockNetworkService("mgmt_fabnet", "FABNetv4",
                                [be_nic2_iface, db_nic_iface], subnet="10.1.0.0/24", gateway="10.1.0.254")

slice_web = MockSlice("web_application", state="StableOK",
                      nodes=[frontend, backend, database],
                      networks=[app_bridge, mgmt_net])

# ── Slice 2: ml_training ──
tr_nic_iface = MockInterface("trainer-nic1-p0", ip="10.2.0.1", mac="aa:bb:cc:00:01:01", bandwidth=100, site="UTAH")
ds_nic_iface = MockInterface("data_server-nic1-p0", ip="10.2.0.2", mac="aa:bb:cc:00:01:02", site="RENC")
gw_nic_iface = MockInterface("gpu_worker-nic1-p0", ip="10.2.0.3", mac="aa:bb:cc:00:01:03", site="UCSD")
fp_iface     = MockInterface("cloudlab-port-p0", vlan="300", site="STAR")

tr_nic = MockComponent("nic1", "NIC_ConnectX_6", [tr_nic_iface])
tr_gpu = MockComponent("gpu1", "GPU_A40", [])
ds_nic = MockComponent("nic1", "NIC_Basic", [ds_nic_iface])
gw_nic = MockComponent("nic1", "NIC_Basic", [gw_nic_iface])
gw_gpu = MockComponent("gpu1", "GPU_A40", [])

trainer     = MockNode("trainer", "UTAH", cores=16, ram=64, disk=200,
                       components=[tr_nic, tr_gpu], interfaces=[tr_nic_iface])
data_server = MockNode("data_server", "RENC", cores=4, ram=16, disk=100,
                       components=[ds_nic], interfaces=[ds_nic_iface])
gpu_worker  = MockNode("gpu_worker", "UCSD", cores=8, ram=32, disk=200,
                       components=[gw_nic, gw_gpu], interfaces=[gw_nic_iface])

tr_nic_iface._node = trainer
ds_nic_iface._node = data_server
gw_nic_iface._node = gpu_worker

cloudlab_fp = MockFacilityPort("cloudlab_port", "STAR", [fp_iface])
compute_net = MockNetworkService("compute_fabnet", "FABNetv4",
                                 [tr_nic_iface, ds_nic_iface, gw_nic_iface],
                                 subnet="10.2.0.0/24", gateway="10.2.0.254")

slice_ml = MockSlice("ml_training", state="StableOK",
                     nodes=[trainer, data_server, gpu_worker],
                     networks=[compute_net],
                     facilities=[cloudlab_fp])

fablib = MockFablib([slice_web, slice_ml])

print(f"Built {len(fablib.get_slices())} slices:")
for s in fablib.get_slices():
    print(f"  {s.get_name()}: {len(s.get_nodes())} nodes, {len(s.get_network_services())} networks")

---
## 1. Interactive Tool — FabVis

Launch the **FABRIC Visualization Suite**. The bottom bar provides all controls in one row:
- **Slice** dropdown + **Load** button — pick a slice and load it into the current view
- **View** dropdown — switch between **Editor** and **Geographic** modes
- **View-specific controls** — Layout/Fit (Editor) or Sites/Links toggles (Geographic)
- **Save** — export the current view as an image

Click any element to inspect it in the detail panel.

In [ ]:
from fabrictestbed_extensions.fabvis import FabVis

fv = FabVis(fablib)
fv.show()

---
## 2. Static Image — Graph Topology

Generate a publication-quality topology diagram from a slice object.
No interactive widget needed — works headlessly for automated reporting, papers, or batch processing.

In [ ]:
from fabrictestbed_extensions.fabvis import render_slice_graph

# Render a single slice
fig = render_slice_graph(slice_web)
fig

In [ ]:
# Multiple slices in one image, saved to file
fig = render_slice_graph(
    [slice_web, slice_ml],
    save="all_slices_topology.png",
    title="All FABRIC Experiments",
    figsize=(18, 12),
    dpi=300,
)
fig

---
## 3. Static Image — Geographic Map

Generate a geographic view showing nodes at their FABRIC sites with network links.

In [ ]:
from fabrictestbed_extensions.fabvis import render_slice_map

# Render a single slice on the map
fig = render_slice_map(slice_ml)
fig

In [ ]:
# Save as PDF (vector format, great for papers)
fig = render_slice_map(
    [slice_web, slice_ml],
    save="fabric_geo.pdf",
    title="FABRIC Experiment Deployment",
)

import matplotlib.pyplot as plt
plt.close("all")
print("Saved: fabric_geo.pdf")

---
## Quick Reference

```python
from fabrictestbed_extensions.fablib.fablib import FablibManager
from fabrictestbed_extensions.fabvis import FabVis, render_slice_graph, render_slice_map

fablib = FablibManager()

# Interactive tool
fv = FabVis(fablib)
fv.show()

# Static images (no GUI)
render_slice_graph("my_slice", fablib=fablib, save="topology.png")
render_slice_map("my_slice", fablib=fablib, save="geo.png")
```